# 02 — End-to-End ML Pipeline
## Healthcare Pathways AI | Day 1 — Foundational ML

This notebook runs the **complete Day 1 pipeline** from raw data to trained models
and evaluation report. It calls the `src/` modules directly, keeping the notebook
concise and the business logic reusable.

**Pipeline stages:**
1. Generate synthetic data → `data/raw/patients_raw.csv`
2. Preprocess & feature-engineer → `data/processed/`
3. Train clustering + classification models → `models/`
4. Evaluate and generate report → `reports/`


In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import joblib

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

ROOT = os.path.abspath("..")
PROC_DIR = os.path.join(ROOT, "data", "processed")
MODELS_DIR = os.path.join(ROOT, "models")
REPORTS_DIR = os.path.join(ROOT, "reports")


## Step 1 — Generate Synthetic Dataset


In [ ]:
from src.generate_data import generate_patients

df_raw = generate_patients(n=6000, seed=42)
os.makedirs(os.path.join(ROOT, "data", "raw"), exist_ok=True)
df_raw.to_csv(os.path.join(ROOT, "data", "raw", "patients_raw.csv"), index=False)

print(f"Dataset shape   : {df_raw.shape}")
print(f"Positive class  : {df_raw['high_risk_complication'].mean():.2%}")
print(f"Missing cells   : {df_raw.isnull().sum().sum()}")
df_raw.head(3)


## Step 2 — Preprocessing & Feature Engineering


In [ ]:
from src.data_prep import run_preprocessing
run_preprocessing()


In [ ]:
# Verify outputs
X_train = pd.read_csv(os.path.join(PROC_DIR, "X_train.csv"))
X_test  = pd.read_csv(os.path.join(PROC_DIR, "X_test.csv"))
y_train = pd.read_csv(os.path.join(PROC_DIR, "y_train.csv")).values.ravel()
y_test  = pd.read_csv(os.path.join(PROC_DIR, "y_test.csv")).values.ravel()
df_imp  = pd.read_csv(os.path.join(PROC_DIR, "patients_imputed.csv"))

with open(os.path.join(PROC_DIR, "feature_names.txt")) as f:
    feature_names = [l.strip() for l in f]

print(f"X_train: {X_train.shape}  X_test: {X_test.shape}")
print(f"Features: {len(feature_names)}")
print(f"Derived features added: bmi_category, comorbidity_count, risk_adjusted_adherence, metabolic_syndrome_proxy")


## Step 3A — Patient Segmentation (Clustering)


In [ ]:
from src.train import run_clustering

X_all = np.vstack([X_train.values, X_test.values])
optimal_k = run_clustering(X_all, df_imp, feature_names)
print(f"\nOptimal k = {optimal_k}")


In [ ]:
# Visualise elbow / silhouette
elbow_df = pd.read_csv(os.path.join(PROC_DIR, "elbow_data.csv"))

fig, ax1 = plt.subplots(figsize=(9, 5))
ax2 = ax1.twinx()

ax1.plot(elbow_df["k"], elbow_df["inertia"], "bo-", lw=2, markersize=7, label="Inertia (Elbow)")
ax2.plot(elbow_df["k"], elbow_df["silhouette"], "rs--", lw=2, markersize=7, label="Silhouette Score")

best_k = elbow_df.loc[elbow_df["silhouette"].idxmax(), "k"]
ax2.axvline(best_k, color="gray", linestyle=":", linewidth=1.5, label=f"Optimal k={best_k}")

ax1.set_xlabel("k (Number of Clusters)"); ax1.set_ylabel("Inertia", color="blue")
ax2.set_ylabel("Silhouette Score", color="red")
ax1.tick_params(axis="y", labelcolor="blue"); ax2.tick_params(axis="y", labelcolor="red")

lines = ax1.get_legend_handles_labels()[0] + ax2.get_legend_handles_labels()[0]
labels = ax1.get_legend_handles_labels()[1] + ax2.get_legend_handles_labels()[1]
ax1.legend(lines, labels, loc="upper right", fontsize=9)

plt.title(f"K-Means Elbow & Silhouette Analysis → Optimal k = {best_k}", fontsize=12, fontweight="bold")
plt.tight_layout(); plt.show()


In [ ]:
# Cluster profiles
cluster_profiles = pd.read_csv(os.path.join(PROC_DIR, "cluster_profiles.csv"))
display_cols = ["cluster_kmeans", "cluster_label", "n_patients",
                "avg_age", "avg_hba1c", "avg_adherence", "risk_rate"]
print(cluster_profiles[display_cols].to_string(index=False))


In [ ]:
# Cluster heatmap
profile_cols = ["avg_age", "avg_bmi", "avg_hba1c", "avg_fasting_glucose",
                "avg_systolic_bp", "avg_adherence", "avg_genetic_risk", "risk_rate"]

plot_df = cluster_profiles.set_index("cluster_label")[profile_cols].copy()
plot_norm = (plot_df - plot_df.min()) / (plot_df.max() - plot_df.min() + 1e-9)

fig, ax = plt.subplots(figsize=(12, 4))
sns.heatmap(plot_norm.T, annot=plot_df.T.round(2), fmt="g",
            cmap="RdYlGn_r", ax=ax, linewidths=0.5,
            cbar_kws={"label": "Normalised value"})
ax.set_title("Cluster Profile Heatmap (Normalised; annotated with raw means)",
             fontsize=11, fontweight="bold")
ax.tick_params(axis="x", rotation=20)
plt.tight_layout(); plt.show()


## Step 3B — Risk Prediction (Classification)


In [ ]:
from src.train import run_classification

all_results = run_classification(
    X_train.values, X_test.values, y_train, y_test, feature_names
)


In [ ]:
# Results comparison table
model_comparison = pd.read_csv(os.path.join(PROC_DIR, "model_comparison.csv"))
print("\nModel Comparison:")
print(model_comparison.to_string(index=False))

best = model_comparison.loc[model_comparison["roc_auc"].idxmax()]
print(f"\n→ Best model: {best['model']} (AUC = {best['roc_auc']:.4f})")


In [ ]:
# ROC curves
from sklearn.metrics import roc_curve, auc as sk_auc

fig, ax = plt.subplots(figsize=(8, 6))
colors = ["#2196F3", "#4CAF50", "#FF5722"]
model_files = ["logistic_regression", "random_forest", "gradient_boosting"]
model_names = ["Logistic Regression", "Random Forest", "Gradient Boosting"]

for fname, name, color in zip(model_files, model_names, colors):
    path = os.path.join(MODELS_DIR, f"{fname}.joblib")
    if os.path.exists(path):
        model = joblib.load(path)
        y_prob = model.predict_proba(X_test.values)[:, 1]
        fpr, tpr, _ = roc_curve(y_test, y_prob)
        roc_auc = sk_auc(fpr, tpr)
        ax.plot(fpr, tpr, color=color, lw=2, label=f"{name} (AUC={roc_auc:.3f})")

ax.plot([0,1],[0,1],"k--",lw=1,label="Random")
ax.set(xlabel="False Positive Rate", ylabel="True Positive Rate",
       title="ROC Curve Comparison — Complication Risk Prediction")
ax.legend(loc="lower right"); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()


In [ ]:
# Confusion matrices
from sklearn.metrics import confusion_matrix

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, fname, name in zip(axes, model_files, model_names):
    path = os.path.join(MODELS_DIR, f"{fname}.joblib")
    if os.path.exists(path):
        model = joblib.load(path)
        cm = confusion_matrix(y_test, model.predict(X_test.values))
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                    xticklabels=["Low","High"], yticklabels=["Low","High"],
                    annot_kws={"size": 14})
        ax.set_title(name, fontsize=10, fontweight="bold")
        ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
plt.suptitle("Confusion Matrices — Test Set", fontsize=12, fontweight="bold")
plt.tight_layout(); plt.show()


In [ ]:
# Feature importance
img_path = os.path.join(REPORTS_DIR, "feature_importance.png")
if os.path.exists(img_path):
    from IPython.display import Image
    display(Image(img_path, width=900))


## Step 4 — Evaluate & Generate Report


In [ ]:
from src.evaluate import main as eval_main
eval_main()


In [ ]:
# Show report path
report_path = os.path.join(REPORTS_DIR, "model_evaluation_report.md")
print(f"[OK] Report written to: {report_path}")

# Show final model files
print("\nSaved model files:")
for f in sorted(os.listdir(MODELS_DIR)):
    path = os.path.join(MODELS_DIR, f)
    size = os.path.getsize(path) / 1e3
    print(f"  {f:40s} {size:8.1f} KB")


## Pipeline Summary

| Stage | Output | Status |
|---|---|---|
| Data generation | `data/raw/patients_raw.csv` | ✅ |
| Preprocessing | `data/processed/X_train/test, y_train/test` | ✅ |
| Feature engineering | bmi_category, comorbidity_count, risk_adjusted_adherence, metabolic_syndrome | ✅ |
| K-Means clustering | `models/kmeans_model.joblib`, cluster profiles | ✅ |
| Hierarchical clustering | `models/hierarchical_model.joblib` | ✅ |
| Logistic Regression | `models/logistic_regression.joblib` | ✅ |
| Random Forest | `models/random_forest.joblib` | ✅ |
| Gradient Boosting | `models/gradient_boosting.joblib` | ✅ |
| Best model saved | `models/best_risk_model.joblib` | ✅ |
| Evaluation report | `reports/model_evaluation_report.md` | ✅ |

---
**Next:** Day 2 — Deep Learning Risk Model (do not implement yet)
